<a href="https://colab.research.google.com/github/luciaPi/MLSS2026-generative-models/blob/main/5_Stable_Diffusion_cisty.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stable Diffusion (text-to-image generation)



## 1. Inštalácia knižníc

In [ ]:
!pip install -q diffusers transformers accelerate safetensors
print("Knižnice nainštalované!")

In [ ]:
import torch
from diffusers import StableDiffusionPipeline, StableDiffusionImg2ImgPipeline
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Používam zariadenie: {device}")

## 2. Načítanie Stable Diffusion

In [ ]:
print("Načítavam Stable Diffusion 1.5...")
print(" Prvé spustenie: Stiahne model (chvíľu to trvá!)")
print("")

model_id = "runwayml/stable-diffusion-v1-5"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,  # FP16 pre rýchlosť (half precision)
    safety_checker=None  # Vypneme safety checker pre demo
)
pipe = pipe.to(device)

# Optimalizácia pamäte
pipe.enable_attention_slicing()

print("\n Stable Diffusion načítaný!")

## 3. Helper funkcie

In [ ]:
def generate_image(prompt, negative_prompt="", num_inference_steps=50,
                   guidance_scale=7.5, seed=None):
    """
    Vygeneruje obrázok z text promptu

    Args:
        prompt: Text prompt
        negative_prompt: Čo NECHCEME v obrázku
        num_inference_steps: Počet denoising krokov (50 = štandard) (kvalita vs. rýchlosť)
        guidance_scale: CFG scale (7.5 = štandard) (ako silno má model sledovať text)
        seed: Random seed pre reprodukovateľnosť
    """
    generator = None if seed is None else torch.Generator(device=device).manual_seed(seed)

    with torch.no_grad():
        image = pipe(   # volanie Stable Diffusion
            prompt=prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator
        ).images[0]

    return image # vygenerovane obrazky

def show_image(image, title=""):
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.title(title, fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_grid(images, titles, cols=3):
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*5))
    axes = axes.flatten() if len(images) > 1 else [axes]

    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img)
        ax.set_title(title, fontsize=12)
        ax.axis('off')

    for ax in axes[len(images):]:
        ax.axis('off')

    plt.tight_layout()
    plt.show()

print("Helper funkcie pripravené!")

## 4. Základné text-to-image

In [ ]:
print("="*60)
print("ZÁKLADNÉ TEXT-TO-IMAGE GENEROVANIE")
print("="*60)

prompt = "a beautiful mountain landscape at sunset, highly detailed, 8k"
print(f"Prompt: {prompt}\n")

image = generate_image(prompt, seed=42)
show_image(image, f"Generated: {prompt}")

## 5. Negative Prompts

In [ ]:
print("="*60)
print("NEGATIVE PROMPTS EXPERIMENT")
print("="*60)

prompt = "a cat sitting on a table"
seed = 123

# Bez negative prompt
print("1. Bez negative prompt...")
img1 = generate_image(prompt, negative_prompt="", seed=seed)

# S negative prompt
print("2. S negative prompt...")
negative = "blurry, low quality, distorted, deformed"
img2 = generate_image(prompt, negative_prompt=negative, seed=seed)

show_grid(
    [img1, img2],
    ["Bez negative prompt", f"S negative prompt:\n{negative}"],
    cols=2
)

In [ ]:
print("="*60)
print("NEGATIVE PROMPTS EXPERIMENT")
print("="*60)

prompt = "a cat sitting on a table"
seed = 123

# Bez negative prompt
print("1. Bez negative prompt...")
img1 = generate_image(prompt, negative_prompt="", seed=seed)

# S negative prompt
print("2. S negative prompt...")
negative = "ears"
img2 = generate_image(prompt, negative_prompt=negative, seed=seed)

show_grid(
    [img1, img2],
    ["Bez negative prompt", f"S negative prompt:\n{negative}"],
    cols=2
)

In [ ]:
print("="*60)
print("NEGATIVE PROMPTS EXPERIMENT")
print("="*60)

prompt = "a cat sitting on a table"
seed = 123

# Bez negative prompt
print("1. Bez negative prompt...")
img1 = generate_image(prompt, negative_prompt="", seed=seed)

# S negative prompt
print("2. S negative prompt...")
negative = "cat"
img2 = generate_image(prompt, negative_prompt=negative, seed=seed)

show_grid(
    [img1, img2],
    ["Bez negative prompt", f"S negative prompt:\n{negative}"],
    cols=2
)

## 6. CFG Scale Experimentovanie

In [ ]:
print("="*60)
print("CFG SCALE TESTING")
print("="*60)
print("CFG scale = Classifier-Free Guidance")
print("- Nízke (1-3): Kreatívnejšie, ignoruje prompt")
print("- Stredné (7-8): Balans (štandard)")
print("- Vysoké (15+): Striktne dodržiava prompt")
print("")

prompt = "a golden retriever puppy playing in snow"
seed = 456
cfg_scales = [1.0, 3.0, 7.5, 15.0]

images = []
titles = []

for cfg in cfg_scales:
    print(f"Generujem s CFG={cfg}...")
    img = generate_image(prompt, guidance_scale=cfg, seed=seed)
    images.append(img)
    titles.append(f"CFG = {cfg}")

show_grid(images, titles, cols=2)

In [ ]:
print("="*60)
print("CFG SCALE TESTING")
print("="*60)
print("CFG scale = Classifier-Free Guidance")
print("- Nízke (1-3): Kreatívnejšie, ignoruje prompt")
print("- Stredné (7-8): Balans (štandard)")
print("- Vysoké (15+): Striktne dodržiava prompt")
print("")

prompt = "students at machine learning summer school"
seed = 456
cfg_scales = [1.0, 3.0, 7.5, 15.0]

images = []
titles = []

for cfg in cfg_scales:
    print(f"Generujem s CFG={cfg}...")
    img = generate_image(prompt, guidance_scale=cfg, seed=seed)
    images.append(img)
    titles.append(f"CFG = {cfg}")

show_grid(images, titles, cols=2)

## 7. Rôzne štýly

In [ ]:
print("="*60)
print("EXPERIMENTOVANIE SO ŠTÝLMI")
print("="*60)

base_prompt = "a castle on a hill"

styles = [
    ("realistic photo", f"{base_prompt}, realistic photo, 8k, highly detailed"),
    ("oil painting", f"{base_prompt}, oil painting, impressionist style"),
    ("cyberpunk", f"{base_prompt}, cyberpunk style, neon lights, futuristic"),
    ("watercolor", f"{base_prompt}, watercolor painting, soft colors"),
    ("pencil sketch", f"{base_prompt}, pencil sketch, black and white drawing"),
    ("anime style", f"{base_prompt}, anime style, manga, cel shaded")
]

images = []
titles = []

for style_name, styled_prompt in styles:
    print(f"Generujem: {style_name}...")
    img = generate_image(styled_prompt)
    images.append(img)
    titles.append(style_name)

show_grid(images, titles, cols=3)

## 8. Grid experimentovanie

In [ ]:
print("="*60)
print("GRID EXPERIMENT: Rovnaký prompt, rôzne seeds")
print("="*60)

prompt = "a cute corgi puppy, professional photo, 8k"
seeds = [100, 200, 300, 400, 500, 600]

images = []
titles = []

for seed in seeds:
    print(f"Seed {seed}...")
    img = generate_image(prompt, seed=seed)
    images.append(img)
    titles.append(f"Seed: {seed}")

show_grid(images, titles, cols=3)

## Záver

**Kľúčové poznatky:**

1. **Stable Diffusion -> Latent Diffusion**
   - Difúzia v komprimovanom priestore (rýchlejšie)
   - VAE encoder/decoder

2. **Text-to-image pipeline:**
   - Text → CLIP → Text embedding
   - Difúzia v latentnom priestore
   - VAE decoder → Obrázok

3. **Dôležité parametre:**
   - **guidance_scale**: Ako veľmi dodržiavať prompt (7.5 = štandard)
   - **negative_prompt**: Čo NECHCEME v obrázku
   - **num_steps**: Kvalita vs rýchlosť (50 = štandard)
   - **seed**: Reprodukovateľnosť

Vytvorené s použitím Claude AI.